# Análisis Semántico con Word2Vec

## Imports y configuración

In [2]:
import warnings, os, re
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.spatial.distance import cosine
from gensim.models import Word2Vec
import sys
sys.path.append(os.path.abspath('../../../../../..'))
from src.data.mongo_storage import _get_default_collection, guardar_embeddings

FIGS   = Path('../../..') / 'data' / 'figures'
MODELS = Path('../../..') / 'data' / 'models'
for d in [FIGS, MODELS]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.grid':True,'grid.alpha':0.3,'font.size':11})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']
print('✓ Imports OK')


✓ Imports OK


## Cargar corpus desde MongoDB

In [3]:
col = _get_default_collection()
docs = list(col.find(
    {'Lyrics': {'$ne': None}},
    {'_id': 1, 'Song': 1, 'Artist': 1, 'Genre': 1, 'Song year': 1, 'Lyrics': 1, 'Language': 1}
))
df = pd.DataFrame(docs)
df = df.dropna(subset=['Lyrics', 'Genre']).copy()
df['Lyrics'] = df['Lyrics'].astype(str)
df = df[df['Lyrics'].str.len() > 50].reset_index(drop=True)

# Solo géneros con >= 20 canciones
generos_validos = df['Genre'].value_counts()
generos_validos = generos_validos[generos_validos >= 20].index.tolist()
df = df[df['Genre'].isin(generos_validos)].reset_index(drop=True)

print(f'✓ {len(df):,} canciones cargadas desde MongoDB | {df["Genre"].nunique()} géneros')
print(df['Genre'].value_counts().to_string())


✓ 7,935 canciones cargadas desde MongoDB | 10 géneros
Genre
Rock          1410
Pop           1110
Hip-Hop        960
Country        810
Metal          810
Jazz           660
Electronic     660
Indie          510
R&B            510
Folk           495


## Tokenización

In [ ]:
STOPWORDS = {
    'the','a','an','is','it','in','of','to','and','or','i','me','my','you',
    'your','we','he','she','they','them','this','that','was','were','be',
    'been','have','has','had','do','did','will','would','could','should',
    'not','no','so','but','if','at','on','for','with','as','by','from',
}

def tokenizar(texto: str) -> list:
    tokens = re.findall(r'[a-z]+', texto.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

print(f'Tokenizando {len(df)} canciones...')
df['tokens_w2v'] = df['Lyrics'].apply(tokenizar)
corpus_tokens = df['tokens_w2v'].tolist()
print(f'✓ {sum(len(t) for t in corpus_tokens):,} tokens totales')


## Entrenar Word2Vec — CBOW y Skip-Gram

In [ ]:
print('Entrenando Word2Vec CBOW...')
w2v_cbow = Word2Vec(
    corpus_tokens, vector_size=100, window=5, min_count=3,
    sg=0, epochs=15, seed=42, workers=4,
)
print(f'  ✓ CBOW      — vocabulario: {len(w2v_cbow.wv):,} palabras')

print('Entrenando Word2Vec Skip-Gram...')
w2v_sg = Word2Vec(
    corpus_tokens, vector_size=100, window=5, min_count=3,
    sg=1, epochs=15, seed=42, workers=4,
)
print(f'  ✓ Skip-Gram — vocabulario: {len(w2v_sg.wv):,} palabras')

w2v_cbow.save(str(MODELS / 'w2v_cbow.model'))
w2v_sg.save(str(MODELS / 'w2v_skipgram.model'))
print('\n✓ Modelos guardados en disco')


## Guardar embeddings Word2Vec en MongoDB

In [ ]:
def w2v_promedio(tokens: list, modelo: Word2Vec) -> list:
    vecs = [modelo.wv[t] for t in tokens if t in modelo.wv]
    return np.mean(vecs, axis=0).tolist() if vecs else [0.0] * modelo.vector_size

actualizados = 0
for _, row in df.iterrows():
    emb = w2v_promedio(row['tokens_w2v'], w2v_sg)   # Skip-Gram como representativo
    ok = guardar_embeddings(
        song_id=row['_id'],
        word2vec_avg=emb,
        beto_cls=[],   # se rellenará en el notebook BETO
    )
    if ok:
        actualizados += 1

print(f'✓ word2vec_avg guardado en MongoDB: {actualizados}/{len(df)} canciones')


## Exploración de campos semánticos

In [ ]:
print('=== Vecinos semánticos más cercanos (Skip-Gram) ===\n')
palabras_clave = ['love', 'fight', 'night', 'money', 'soul', 'dark', 'road', 'fire']
for palabra in palabras_clave:
    if palabra in w2v_sg.wv:
        vecinos = w2v_sg.wv.most_similar(palabra, topn=6)
        print(f'  {palabra:<10} → {', '.join([f"{w}({s:.2f})" for w, s in vecinos])}')
    else:
        print(f'  {palabra:<10} → no en vocabulario')


## Analogías vectoriales

In [ ]:
print('=== Analogías vectoriales (A - B + C = ?) ===\n')
analogias = [
    (['rock', 'guitar'],   ['piano'],   'rock + guitar - piano'),
    (['love', 'happy'],    ['sad'],     'love + happy - sad'),
    (['night', 'dark'],    ['day'],     'night + dark - day'),
    (['street', 'hustle'], ['peace'],   'street + hustle - peace'),
    (['broken', 'heart'],  ['happy'],   'broken + heart - happy'),
    (['jazz', 'trumpet'],  ['guitar'],  'jazz - guitar + trumpet'),
]
for pos, neg, desc in analogias:
    ausentes = [w for w in pos+neg if w not in w2v_sg.wv]
    if ausentes:
        print(f'  {desc} → ⚠ no en vocab: {ausentes}')
        continue
    res = w2v_sg.wv.most_similar(positive=pos, negative=neg, topn=3)
    print(f'  {desc}')
    print(f'    → {" | ".join([f"{w}({s:.2f})" for w, s in res])}')
    print()


## Similitud entre géneros (vectores promedio)

In [ ]:
def vector_genero(genero: str, modelo: Word2Vec) -> np.ndarray:
    letras = df[df['Genre'] == genero]['tokens_w2v'].tolist()
    vecs = []
    for tokens in letras:
        tvecs = [modelo.wv[t] for t in tokens if t in modelo.wv]
        if tvecs:
            vecs.append(np.mean(tvecs, axis=0))
    return np.mean(vecs, axis=0) if vecs else np.zeros(modelo.vector_size)

generos_corpus = df['Genre'].unique().tolist()
vecs_genero = {g: vector_genero(g, w2v_sg) for g in generos_corpus}

n = len(generos_corpus)
sim_generos = np.zeros((n, n))
for i, g1 in enumerate(generos_corpus):
    for j, g2 in enumerate(generos_corpus):
        sim_generos[i, j] = 1 - cosine(vecs_genero[g1], vecs_genero[g2])

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_generos, cmap='YlOrRd', vmin=0.5, vmax=1.0)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(generos_corpus, rotation=40, ha='right')
ax.set_yticklabels(generos_corpus)
ax.set_title('Similitud Coseno entre Géneros\n(vectores promedio Word2Vec Skip-Gram)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Similitud coseno')
for i in range(n):
    for j in range(n):
        color = 'white' if sim_generos[i,j] > 0.85 else 'black'
        ax.text(j, i, f'{sim_generos[i,j]:.2f}', ha='center', va='center', fontsize=8, color=color, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / 'similitud_generos_w2v.png', dpi=130, bbox_inches='tight')
plt.show()

pares = [(generos_corpus[i], generos_corpus[j], sim_generos[i,j])
         for i in range(n) for j in range(i+1, n)]
pares.sort(key=lambda x: -x[2])
print('Top 3 más SIMILARES:')
for g1, g2, s in pares[:3]: print(f'  {g1} ↔ {g2}: {s:.4f}')
print('\nTop 3 más DIFERENTES:')
for g1, g2, s in pares[-3:]: print(f'  {g1} ↔ {g2}: {s:.4f}')
